# AIC 2026 — Kiểm tra tính toàn vẹn dữ liệu (Data Integrity Check)

**Mục đích notebook này:** sau nhiều lần gặp sự cố (Keyframes giải nén thiếu, cấu trúc thư mục lồng nhau, cột dữ liệu không khớp giả định...), notebook này **đối chiếu chéo TẤT CẢ nguồn dữ liệu** trong Index Store với nhau, để phát hiện sớm bất kỳ chỗ nào **thiếu sót hoặc không khớp** — TRƯỚC khi chạy Pipeline Online thật, tránh phát hiện lỗi giữa chừng như đã từng gặp.

**Nguyên tắc kiểm tra:** dùng `clip_mapping_df` (từ CLIP features BTC cấp) làm **danh sách chuẩn** — vì đây là dữ liệu gốc đảm bảo phủ đủ 100% video trong kho. Mọi nguồn khác (Objects, Text, Map-keyframes, file Keyframes/Video thật...) đều được **đối chiếu ngược lại** với danh sách chuẩn này.

**Cách đọc kết quả:** mỗi phần đều kết thúc bằng dòng `[PASS]` (ổn) hoặc `[CẢNH BÁO]` (cần xem lại) — Phần cuối cùng (Phần 9) gom lại thành 1 bảng tổng kết duy nhất.

---

## Phần 0 — Setup

**Cell này là gì:** gắn Google Drive vào Colab và cài các thư viện cần dùng (`faiss-cpu` để đọc FAISS Index, `pyarrow` để đọc file Parquet).

**Dùng để làm gì:** đây là bước chuẩn bị bắt buộc trước khi đọc bất kỳ dữ liệu nào — không có bước này, các cell sau sẽ báo lỗi thiếu thư viện hoặc không truy cập được Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install faiss-cpu pyarrow --quiet

**Cell này là gì:** khai báo đường dẫn `DATASET_ROOT`/`EXTRACTED_ROOT` — SỬA LẠI cho khớp đúng cấu trúc Drive của bạn (giống 2 notebook Offline/Online đã dùng).

**Dùng để làm gì:** toàn bộ notebook dựa vào 2 biến này để tìm đúng vị trí dữ liệu — sai đường dẫn ở đây sẽ khiến mọi kiểm tra phía sau báo sai lệch dù dữ liệu thực tế không có vấn đề gì.

In [ ]:
import os, json, pickle
from pathlib import Path
import numpy as np
import pandas as pd
import faiss

DATASET_ROOT = "/content/drive/MyDrive/AIC 2026/dataset"
EXTRACTED_ROOT = os.path.join(DATASET_ROOT, "extracted")

print("DATASET_ROOT  :", DATASET_ROOT)
print("EXTRACTED_ROOT:", EXTRACTED_ROOT)
print("Tồn tại:", os.path.exists(EXTRACTED_ROOT))

---
## Phần 1 — Load toàn bộ nguồn dữ liệu (giống Pipeline Online Phần 1)

**Cell này là gì:** đọc lại đúng 6 nguồn dữ liệu đã build ở Pipeline Offline: FAISS Index (CLIP), bảng Object, BM25+Text Index (OCR/ASR), bảng Map-keyframes, bảng Media-info.

**Dùng để làm gì:** đây là "nguyên liệu" cho mọi phép đối chiếu ở các Phần sau — nếu 1 trong các file này thiếu, cell sẽ báo rõ ngay tại đây thay vì lỗi mơ hồ về sau.

In [ ]:
def safe_load(label, load_fn):
    """Load 1 nguồn dữ liệu, KHÔNG làm dừng cả notebook nếu thiếu — chỉ báo rõ
    và trả về None, để các Phần sau tự xử lý phần thiếu này."""
    try:
        result = load_fn()
        print(f"[OK]    {label}")
        return result
    except Exception as e:
        print(f"[THIẾU] {label} -> {e}")
        return None


clip_index = safe_load("CLIP FAISS Index", lambda: faiss.read_index(
    os.path.join(EXTRACTED_ROOT, "clip_index", "clip_faiss.index")))
clip_mapping_df = safe_load("CLIP Mapping", lambda: pd.read_parquet(
    os.path.join(EXTRACTED_ROOT, "clip_index", "clip_mapping.parquet")))
objects_df = safe_load("Objects Table", lambda: pd.read_parquet(
    os.path.join(EXTRACTED_ROOT, "objects", "objects_index.parquet")))

bm25_path = os.path.join(EXTRACTED_ROOT, "final_index", "bm25.pkl")
text_index_path = os.path.join(EXTRACTED_ROOT, "final_index", "text_index.parquet")
text_index_df = safe_load("Text Index (OCR+ASR)", lambda: pd.read_parquet(text_index_path)) \
    if os.path.exists(text_index_path) else pd.DataFrame(columns=["video_id"])

map_keyframes_df = safe_load("Map-keyframes", lambda: pd.read_parquet(
    os.path.join(EXTRACTED_ROOT, "map_keyframes", "map_keyframes_index.parquet")))
media_info_df = safe_load("Media-info", lambda: pd.read_parquet(
    os.path.join(EXTRACTED_ROOT, "media_info", "media_info_index.parquet")))

assert clip_mapping_df is not None, "BẮT BUỘC phải có CLIP Mapping — đây là danh sách chuẩn dùng xuyên suốt notebook này. Dừng lại kiểm tra Offline trước khi tiếp tục."

---
## Phần 2 — Đối chiếu tập `video_id` giữa các nguồn

**Cell này là gì:** với mỗi nguồn dữ liệu, lấy ra tập hợp `video_id` xuất hiện trong đó, rồi so với danh sách chuẩn (CLIP Mapping).

**Dùng để làm gì:** phát hiện ngay lập tức nếu 1 nguồn nào đó (VD Objects, Media-info) **thiếu hẳn** 1 số video so với chuẩn — đây chính là loại lỗi đã gặp thực tế với thư mục Keyframes trước đó, giờ kiểm tra luôn cho TẤT CẢ nguồn, không chỉ Keyframes.

In [ ]:
master_video_ids = set(clip_mapping_df["video_id"].unique())
print(f"Danh sách CHUẨN (CLIP Mapping): {len(master_video_ids)} video\n")

video_id_sources = {
    "Objects": set(objects_df["video_id"].unique()) if objects_df is not None else set(),
    "Text (OCR+ASR)": set(text_index_df["video_id"].unique()) if len(text_index_df) > 0 else set(),
    "Map-keyframes": set(map_keyframes_df["video_id"].unique()) if map_keyframes_df is not None else set(),
    "Media-info": set(media_info_df["video_id"].unique()) if media_info_df is not None else set(),
}

video_id_check_results = {}
for name, vid_set in video_id_sources.items():
    missing = master_video_ids - vid_set
    extra = vid_set - master_video_ids
    status = "PASS" if not missing else "CẢNH BÁO"
    print(f"[{status:9}] {name:16} -> có {len(vid_set)}/{len(master_video_ids)} video "
          f"| thiếu: {len(missing)} | dư thừa lạ: {len(extra)}")
    if missing:
        print(f"             5 video mẫu bị thiếu: {sorted(missing)[:5]}")
    video_id_check_results[name] = {"missing": missing, "extra": extra}

---
## Phần 3 — Kiểm tra file Keyframes THẬT có tồn tại và ĐỦ SỐ LƯỢNG không

**Cell này là gì:** quét trực tiếp thư mục `extracted/keyframes/` trên Drive (không dựa vào bảng dữ liệu nào), đếm xem mỗi video có bao nhiêu file ảnh thật, so với số lượng KỲ VỌNG (số dòng của video đó trong `clip_mapping_df`).

**Dùng để làm gì:** đây chính là loại kiểm tra đã cứu bạn ở sự cố Keyframes trước đó — phát hiện không chỉ "thiếu hẳn video" mà cả trường hợp **có thư mục nhưng RỖNG hoặc THIẾU MỘT PHẦN** ảnh bên trong (lỗi tinh vi hơn, dễ bị bỏ sót nếu chỉ kiểm tra tên thư mục).

In [ ]:
keyframes_root = Path(EXTRACTED_ROOT, "keyframes")

# Dùng rglob để tìm ĐỆ QUY (xử lý đúng cả trường hợp có thêm 1 cấp thư mục lồng,
# như đã gặp thực tế: extracted/keyframes/keyframes/{video_id}/)
all_keyframe_dirs = {p.name: p for p in keyframes_root.rglob("*") if p.is_dir()}

expected_counts = clip_mapping_df.groupby("video_id").size().to_dict()

missing_entirely = []
incomplete = []
ok_count = 0

for video_id, expected_n in expected_counts.items():
    video_dir = all_keyframe_dirs.get(video_id)
    if video_dir is None:
        missing_entirely.append(video_id)
        continue
    actual_n = sum(1 for f in video_dir.glob("*") if f.is_file())
    if actual_n < expected_n:
        incomplete.append({"video_id": video_id, "expected": expected_n, "actual": actual_n})
    else:
        ok_count += 1

print(f"[PASS]    Đủ hoàn toàn: {ok_count}/{len(expected_counts)} video")
print(f"[CẢNH BÁO] Thiếu HẲN thư mục: {len(missing_entirely)} video")
if missing_entirely:
    print(f"           5 mẫu: {missing_entirely[:5]}")
print(f"[CẢNH BÁO] Có thư mục nhưng THIẾU MỘT PHẦN ảnh: {len(incomplete)} video")
if incomplete:
    print(f"           5 mẫu: {incomplete[:5]}")

keyframes_missing_video_ids = set(missing_entirely) | {r["video_id"] for r in incomplete}

---
## Phần 4 — Kiểm tra file Video THẬT có tồn tại không

**Cell này là gì:** quét thư mục `extracted/video/`, kiểm tra mỗi video trong danh sách chuẩn có đúng 1 file `.mp4` tương ứng không.

**Dùng để làm gì:** Pipeline Online (Reranker, TRAKE, hiển thị ảnh) phụ thuộc trực tiếp vào file video thật (qua `find_video_file()`) — nếu thiếu file video, các bước này sẽ âm thầm trả về `None` mà không báo lỗi rõ ràng, nên cần kiểm tra riêng ở đây.

In [ ]:
video_root = Path(EXTRACTED_ROOT, "video")
all_video_files = {p.stem: p for p in video_root.rglob("*.mp4")}

missing_videos = master_video_ids - set(all_video_files.keys())
zero_byte_videos = [vid for vid, p in all_video_files.items()
                     if vid in master_video_ids and p.stat().st_size == 0]

status = "PASS" if not missing_videos and not zero_byte_videos else "CẢNH BÁO"
print(f"[{status}] File video: có {len(all_video_files)} file, "
      f"thiếu {len(missing_videos)}, dung lượng 0 byte (hỏng) {len(zero_byte_videos)}")
if missing_videos:
    print(f"  5 video mẫu bị thiếu file: {sorted(missing_videos)[:5]}")
if zero_byte_videos:
    print(f"  5 video mẫu dung lượng 0 byte: {zero_byte_videos[:5]}")

---
## Phần 5 — Kiểm tra Map-keyframes có KHỚP SỐ LƯỢNG với CLIP Mapping không

**Cell này là gì:** với mỗi video, so sánh SỐ DÒNG trong `map_keyframes_df` (bảng mô tả frame_id thật) với SỐ DÒNG trong `clip_mapping_df` (bảng vector CLIP) — 2 bảng này PHẢI có cùng số lượng keyframe cho mỗi video, vì cùng mô tả 1 tập keyframe.

**Dùng để làm gì:** nếu 2 bảng lệch nhau (VD CLIP có 101 vector nhưng Map-keyframes chỉ có 95 dòng cho cùng 1 video), việc tra cứu `get_real_frame_id()` sẽ trả về `None` cho các keyframe dư ra — đây là loại lỗi **âm thầm**, không phát hiện bằng cách xem tổng số dòng toàn bộ 2 bảng (177321 = 177321, như đã thấy trước đó), mà PHẢI kiểm tra riêng TỪNG VIDEO mới thấy được.

In [ ]:
clip_counts_per_video = clip_mapping_df.groupby("video_id").size()
mk_counts_per_video = map_keyframes_df.groupby("video_id").size()

compare_df = pd.DataFrame({
    "clip_count": clip_counts_per_video,
    "map_keyframes_count": mk_counts_per_video,
}).fillna(0).astype(int)
compare_df["diff"] = compare_df["clip_count"] - compare_df["map_keyframes_count"]

mismatched = compare_df[compare_df["diff"] != 0]
status = "PASS" if len(mismatched) == 0 else "CẢNH BÁO"
print(f"[{status}] Số video LỆCH số lượng giữa CLIP và Map-keyframes: {len(mismatched)}/{len(compare_df)}")
if len(mismatched) > 0:
    print("\n5 dòng lệch nhiều nhất:")
    print(mismatched.reindex(mismatched["diff"].abs().sort_values(ascending=False).index).head(5))

---
## Phần 6 — Kiểm tra thứ tự `frame_idx` trong Map-keyframes (phát hiện dữ liệu bị xáo trộn)

**Cell này là gì:** với 1 mẫu ngẫu nhiên các video, kiểm tra xem cột `frame_idx` (frame thật) có TĂNG DẦN theo đúng thứ tự cột `n` (keyframe thứ mấy) hay không.

**Dùng để làm gì:** đây là kiểm tra "tính hợp lý" (sanity check) — nếu `frame_idx` KHÔNG tăng dần đều theo `n` (VD keyframe thứ 5 lại có frame_idx NHỎ HƠN keyframe thứ 4), rất có thể dữ liệu đã bị xáo trộn dòng ở đâu đó trong quá trình xử lý Offline, cần điều tra thêm trước khi tin dùng.

In [ ]:
import random

sample_video_ids = random.sample(list(master_video_ids), min(30, len(master_video_ids)))
not_monotonic = []

for vid in sample_video_ids:
    rows = map_keyframes_df[map_keyframes_df["video_id"] == vid].sort_values("n")
    if len(rows) < 2:
        continue
    frame_idx_values = rows["frame_idx"].values
    if not all(frame_idx_values[i] <= frame_idx_values[i+1] for i in range(len(frame_idx_values)-1)):
        not_monotonic.append(vid)

status = "PASS" if not not_monotonic else "CẢNH BÁO"
print(f"[{status}] Kiểm tra {len(sample_video_ids)} video mẫu -> "
      f"{len(not_monotonic)} video có frame_idx KHÔNG tăng dần đều theo n")
if not_monotonic:
    print(f"  Video cần điều tra thêm: {not_monotonic}")

---
## Phần 7 — Kiểm tra bảng Objects (phạm vi giá trị, video lạ)

**Cell này là gì:** kiểm tra 3 điều với bảng Object Detection — (1) cột `confidence` có nằm đúng trong khoảng [0,1] không (giá trị ngoài khoảng này là dấu hiệu lỗi đọc dữ liệu), (2) có `video_id` nào xuất hiện trong Objects nhưng KHÔNG có trong danh sách chuẩn không (dữ liệu "mồ côi", có thể do lẫn dữ liệu batch khác), (3) tổng quan phân bố nhãn.

**Dùng để làm gì:** phát hiện lỗi xử lý dữ liệu tinh vi hơn — không phải "thiếu" mà là "giá trị sai" hoặc "lẫn dữ liệu không thuộc kho hiện tại".

In [ ]:
invalid_confidence = objects_df[(objects_df["confidence"] < 0) | (objects_df["confidence"] > 1)]
orphan_object_videos = set(objects_df["video_id"].unique()) - master_video_ids

print(f"Tổng số dòng Objects: {len(objects_df)}")
print(f"Khoảng confidence thực tế: [{objects_df['confidence'].min():.4f}, {objects_df['confidence'].max():.4f}]")
print(f"[{'PASS' if len(invalid_confidence)==0 else 'CẢNH BÁO'}] Dòng có confidence NGOÀI [0,1]: {len(invalid_confidence)}")
print(f"[{'PASS' if len(orphan_object_videos)==0 else 'CẢNH BÁO'}] video_id LẠ (không có trong danh sách chuẩn): {len(orphan_object_videos)}")
if orphan_object_videos:
    print(f"  5 mẫu: {sorted(orphan_object_videos)[:5]}")

print("\nTop 10 nhãn phổ biến nhất (kiểm tra nhanh có hợp lý không):")
print(objects_df["label"].value_counts().head(10))

---
## Phần 8 — Kiểm tra FAISS Index có khớp số lượng với bảng Mapping không

**Cell này là gì:** so sánh `clip_index.ntotal` (tổng số vector THẬT SỰ nằm trong FAISS Index) với số dòng của `clip_mapping_df` (bảng ghi chú "vector nào ứng với video/frame nào") — 2 con số này BẮT BUỘC phải bằng nhau tuyệt đối.

**Dùng để làm gì:** nếu 2 con số lệch nhau, nghĩa là khi build FAISS Index ở Offline, có video bị xử lý 2 lần (dư vector) hoặc bảng mapping bị ghi thiếu — dẫn đến toàn bộ kết quả `visual_search()` sẽ tra sai `video_id`/`frame_id` cho MỌI truy vấn (lỗi nghiêm trọng nhất có thể xảy ra, vì Visual Search là module quan trọng nhất).

In [ ]:
n_vectors = clip_index.ntotal
n_mapping_rows = len(clip_mapping_df)

status = "PASS" if n_vectors == n_mapping_rows else "CẢNH BÁO NGHIÊM TRỌNG"
print(f"[{status}] Số vector trong FAISS Index: {n_vectors}")
print(f"[{status}] Số dòng trong CLIP Mapping : {n_mapping_rows}")
if n_vectors != n_mapping_rows:
    print("  -> LỆCH NHAU! Visual Search sẽ tra SAI video_id/frame_id cho mọi kết quả.")
    print("  -> Cần chạy lại Offline Phần 2 (CLIP -> FAISS Index) từ đầu, không dùng fallback\n"
          "     'đã có sẵn thì dùng thẳng' cho lần chạy lại này.")

---
## Phần 9 — Tổng kết toàn bộ kết quả kiểm tra

**Cell này là gì:** gom lại TẤT CẢ kết quả từ Phần 2-8 thành 1 bảng duy nhất, dễ nhìn tổng quan.

**Dùng để làm gì:** đây là "báo cáo cuối cùng" — đọc riêng Phần này là đủ biết dữ liệu có sẵn sàng để chạy Pipeline Online hay chưa, không cần đọc lại chi tiết từng Phần ở trên (trừ khi có `CẢNH BÁO` cần điều tra thêm).

In [ ]:
print("=" * 65)
print("BÁO CÁO TỔNG KẾT — TÍNH TOÀN VẸN DỮ LIỆU")
print("=" * 65)

summary = [
    ("Video_id nhất quán (Objects)", len(video_id_check_results.get("Objects", {}).get("missing", [])) == 0),
    ("Video_id nhất quán (Text OCR+ASR)", len(video_id_check_results.get("Text (OCR+ASR)", {}).get("missing", [])) == 0),
    ("Video_id nhất quán (Map-keyframes)", len(video_id_check_results.get("Map-keyframes", {}).get("missing", [])) == 0),
    ("Video_id nhất quán (Media-info)", len(video_id_check_results.get("Media-info", {}).get("missing", [])) == 0),
    ("File Keyframes đầy đủ", len(keyframes_missing_video_ids) == 0),
    ("File Video đầy đủ", len(missing_videos) == 0 and len(zero_byte_videos) == 0),
    ("Map-keyframes khớp số lượng với CLIP", len(mismatched) == 0),
    ("Frame_idx tăng dần hợp lý", len(not_monotonic) == 0),
    ("Objects: confidence hợp lệ + không có video lạ", len(invalid_confidence) == 0 and len(orphan_object_videos) == 0),
    ("FAISS Index khớp CLIP Mapping", n_vectors == n_mapping_rows),
]

n_pass = sum(1 for _, ok in summary if ok)
for label, ok in summary:
    print(f"  [{'PASS' if ok else 'CẢNH BÁO'}]  {label}")

print("-" * 65)
print(f"KẾT QUẢ: {n_pass}/{len(summary)} mục PASS")
if n_pass == len(summary):
    print("-> Dữ liệu SẴN SÀNG để chạy Pipeline Online.")
else:
    print("-> CÒN CẢNH BÁO — nên xử lý xong các mục trên trước khi tin tưởng\n"
          "   hoàn toàn vào kết quả Pipeline Online (đặc biệt Reranker/QA/TRAKE).")

---
## Phần 10 — Khắc phục tự động (chỉ chạy nếu Phần 3 báo có video thiếu/thiếu 1 phần)

**Cell này là gì:** dùng lại ĐÚNG biến `keyframes_missing_video_ids` đã tính ở Phần 3 (gồm cả video THIẾU HẲN lẫn video có thư mục nhưng THIẾU MỘT PHẦN ảnh) — tìm đúng file zip gốc chứa các video này, giải nén CỤC BỘ trước (nhanh), rồi GHI ĐÈ SẠCH thư mục cũ trên Drive (xóa phần dở dang, thay bằng bản đầy đủ).

**Dùng để làm gì:** khắc phục dứt điểm các video bị lỗi mà không cần đụng đến 870 video đã đủ — an toàn để chạy lại nhiều lần (tự tính lại phần còn thiếu mỗi lần chạy, không phụ thuộc file checkpoint riêng).

In [ ]:
import shutil, zipfile

def extract_missing_videos_selectively(source_zip_folder, dest_folder, missing_video_ids: set,
                                         local_tmp_dir: str = "/content/tmp_selective_extract"):
    """Giải nén CHỈ ĐÚNG các video còn thiếu/thiếu 1 phần — giải nén CỤC BỘ trước
    (nhanh, không phụ thuộc tốc độ ghi Drive), rồi GHI ĐÈ SẠCH thư mục đích trên Drive
    (xóa bản dở dang nếu có, đảm bảo không lẫn file cũ hỏng với file mới)."""
    zip_candidates = sorted(Path(source_zip_folder).glob("*.zip"))
    still_missing = set(missing_video_ids)

    for zip_path in zip_candidates:
        if not still_missing:
            break
        with zipfile.ZipFile(str(zip_path), 'r') as zf:
            members = [n for n in zf.namelist()
                       if any(f"/{vid}/" in n or n.startswith(f"{vid}/") for vid in still_missing)]
            if not members:
                continue
            print(f"  [EXTRACT cục bộ] {len(members)} file từ {zip_path.name}...")
            os.makedirs(local_tmp_dir, exist_ok=True)
            zf.extractall(local_tmp_dir, members=members)

        for vid in list(still_missing):
            local_video_dir = next((p for p in Path(local_tmp_dir).rglob(vid) if p.is_dir()), None)
            if local_video_dir is None:
                continue
            dest_video_dir = Path(dest_folder) / vid
            if dest_video_dir.exists():
                shutil.rmtree(dest_video_dir)   # xóa SẠCH bản dở dang trước khi ghi bản mới
            shutil.copytree(local_video_dir, dest_video_dir)
            still_missing.discard(vid)
            n_files = len(list(dest_video_dir.glob("*")))
            print(f"    [BỔ SUNG] {vid} -> {n_files} file")

        shutil.rmtree(local_tmp_dir, ignore_errors=True)

    if still_missing:
        print(f"  [CẢNH BÁO] Vẫn còn thiếu sau xử lý: {sorted(still_missing)}")
    else:
        print("  Đã khắc phục đầy đủ, không còn video nào thiếu/thiếu 1 phần.")
    return still_missing


if 'keyframes_missing_video_ids' in dir() and keyframes_missing_video_ids:
    print(f"Cần khắc phục {len(keyframes_missing_video_ids)} video: {sorted(keyframes_missing_video_ids)}\n")
    extract_missing_videos_selectively(
        source_zip_folder=os.path.join(DATASET_ROOT, "keyframes"),
        dest_folder=os.path.join(EXTRACTED_ROOT, "keyframes"),
        missing_video_ids=keyframes_missing_video_ids,
    )
else:
    print("Không có video nào cần khắc phục (Phần 3 đã PASS, hoặc chưa chạy Phần 3).")

**Sau khi chạy xong cell trên — chạy lại Phần 3 để xác nhận đã hết `CẢNH BÁO`:**

In [ ]:
# Chạy lại đúng logic Phần 3 để xác nhận đã khắc phục xong
all_keyframe_dirs = {p.name: p for p in keyframes_root.rglob("*") if p.is_dir()}
missing_entirely_recheck = []
incomplete_recheck = []
ok_count_recheck = 0

for video_id, expected_n in expected_counts.items():
    video_dir = all_keyframe_dirs.get(video_id)
    if video_dir is None:
        missing_entirely_recheck.append(video_id)
        continue
    actual_n = sum(1 for f in video_dir.glob("*") if f.is_file())
    if actual_n < expected_n:
        incomplete_recheck.append({"video_id": video_id, "expected": expected_n, "actual": actual_n})
    else:
        ok_count_recheck += 1

status = "PASS" if not missing_entirely_recheck and not incomplete_recheck else "CẢNH BÁO"
print(f"[{status}] Sau khắc phục: {ok_count_recheck}/{len(expected_counts)} video đủ hoàn toàn")
if incomplete_recheck:
    print(f"  Vẫn còn thiếu 1 phần: {incomplete_recheck}")